# Python Lists: Dynamic Arrays, Memory Allocation, and Mutations

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

In Python, the `list` is a **dynamic array** implemented as a contiguous buffer of pointers (`PyObject*`) allocated on the heap. Because it stores pointers rather than contiguous values:
1. Lists can hold heterogeneous objects of arbitrary types.
2. Element access and assignment by index is strictly $O(1)$ arithmetic.
3. CPython employs an over-allocation growth strategy (`0, 4, 8, 16, 24, 32, 40, 52, 64, ...`), ensuring that sequence append operations achieve an **amortized time complexity of $O(1)$**.

### Learning Objectives
1. Inspect list creation patterns and dynamic resizing mechanics.
2. Master forward, backward, and strided slice assignments.
3. Understand time complexities of list mutation operations (`append`, `extend`, `insert`, `pop`, `remove`, `del`).
4. Distinguish between in-place mutation methods (`list.sort()`) and pure functions (`sorted()`).
5. Implement a bounded FIFO buffer using foundational list operations.

In [ ]:
import sys

# List instantiation patterns
empty_list: list[int] = []
numeric_range: list[int] = list(range(10))
replicated_zeros: list[int] = [0] * 8
mixed_types: list[object] = [101, "FeatureVector", 3.14159, True, [1, 2]]

print(f"Empty list byte size: {sys.getsizeof(empty_list)} bytes")
print(f"Range list (10 items): {numeric_range}")
print(f"Replicated zeros:     {replicated_zeros}")
print(f"Heterogeneous list:   {mixed_types}")

assert len(numeric_range) == 10
assert len(replicated_zeros) == 8
assert mixed_types[1] == "FeatureVector"
print("Instantiation assertions passed successfully.")

## Indexing, Slicing, and In-Place Slice Mutation

Python lists support in-place mutation of entire contiguous slices. When assigning to a slice `data[i:j] = new_values`:
- If `len(new_values) == j - i`, elements are replaced in-place without shifting.
- If `len(new_values) != j - i`, the underlying dynamic array automatically shifts elements to expand or contract.
- Assigning an empty list `data[i:j] = []` deletes the specified slice.

In [ ]:
# Slice mutations
data: list[int] = [10, 20, 30, 40, 50, 60, 70]
print(f"Original: {data}")

# Replace slice [2:5] with 2 new elements (contracts the list by 1 element)
data[2:5] = [999, 888]
print(f"After slice replacement [2:5]: {data}")
assert data == [10, 20, 999, 888, 60, 70]

# Delete elements via slice assignment to empty list
data[1:3] = []
print(f"After slice deletion [1:3]:    {data}")
assert data == [10, 888, 60, 70]

# Reversal via slice
reversed_copy: list[int] = data[::-1]
assert reversed_copy == [70, 60, 888, 10]
print("Slice mutation assertions passed successfully.")

## In-Place Mutation Operations and Complexity Profile

| Operation | Syntax | Time Complexity | Memory Effect |
| :--- | :--- | :--- | :--- |
| **Append** | `l.append(x)` | $O(1)$ amortized | Appends reference to end; resizes dynamically |
| **Extend** | `l.extend(it)` | $O(k)$ | Appends $k$ elements in-place |
| **Insert** | `l.insert(i, x)` | $O(n)$ | Shifts all elements from index $i$ to end rightward |
| **Remove** | `l.remove(x)` | $O(n)$ | Searches linearly for first $x$, shifts subsequent elements left |
| **Pop** | `l.pop()` | $O(1)$ | Removes and returns terminal element |
| **Pop Index**| `l.pop(i)` | $O(n - i)$ | Removes element at index $i$, shifts remaining left |
| **Delete** | `del l[i]` | $O(n - i)$ | Deletes element at index $i$ |
| **Clear** | `l.clear()` | $O(1)$ | Empties list and resets size to 0 |
| **Sort** | `l.sort()` | $O(n \log n)$ | In-place stable Timsort |

In [ ]:
# Comprehensive mutation demonstration
numbers: list[int] = [6, 5, 7, 1, 9, 2]

# Append: O(1) amortized
numbers.append(4)
assert numbers[-1] == 4

# Insert at beginning: O(n) shift
numbers.insert(0, 100)
assert numbers[0] == 100

# Pop by index
popped_head = numbers.pop(0)
assert popped_head == 100

# Remove by value (first occurrence)
numbers.remove(7)
assert 7 not in numbers

# Extend
numbers.extend([11, 12])
assert numbers[-2:] == [11, 12]

# In-place reverse
numbers.reverse()
assert numbers[0] == 12

print(f"Final mutated list: {numbers}")
print("All mutation operations validated.")

## In-Place `sort()` vs. Pure `sorted()` Function

- `list.sort(*, key=None, reverse=False)` is an **in-place** operation that mutates the target list and returns `None`.
- `sorted(iterable, *, key=None, reverse=False)` is a **pure function** that accepts any iterable, constructs a new list, sorts it via Timsort, and returns it without altering the source.

In [ ]:
scores: list[int] = [88, 42, 95, 71, 63, 99, 54]

# Pure sorting leaves original untouched
scores_asc: list[int] = sorted(scores)
scores_desc: list[int] = sorted(scores, reverse=True)

assert scores != scores_asc
assert scores_asc[0] == 42 and scores_asc[-1] == 99
assert scores_desc[0] == 99 and scores_desc[-1] == 42

# In-place sorting mutates scores directly
scores.sort()
assert scores == scores_asc

# Key-based sorting (e.g. sorting strings by character length)
words: list[str] = ["convolution", "mlp", "transformer", "rnn", "gan"]
sorted_by_len: list[str] = sorted(words, key=len)
assert (
    sorted_by_len == ["mlp", "rnn", "gan", "convolution", "transformer"]
    or len(sorted_by_len[0]) == 3
)
print("Sorting comparisons validated successfully.")

## Practical Lab Exercise: Bounded FIFO Queue and Order-Preserving Deduplication

Implement:
1. `BoundedQueue[T]`: A class that encapsulates a list as a bounded FIFO queue with `enqueue(item)`, `dequeue()`, `is_full()`, and `is_empty()`.
2. `deduplicate_stable(data: list) -> list`: Deduplicates a list in $O(n)$ average time while preserving original encounter order.

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")


class BoundedQueue(Generic[T]):
    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("Capacity must be strictly positive")
        self._capacity = capacity
        self._storage: list[T] = []

    def enqueue(self, item: T) -> None:
        if len(self._storage) >= self._capacity:
            raise OverflowError("Queue is full")
        self._storage.append(item)

    def dequeue(self) -> T:
        if not self._storage:
            raise IndexError("Queue is empty")
        return self._storage.pop(0)

    def is_full(self) -> bool:
        return len(self._storage) == self._capacity

    def is_empty(self) -> bool:
        return len(self._storage) == 0

    def size(self) -> int:
        return len(self._storage)


def deduplicate_stable(data: list[T]) -> list[T]:
    seen: set[T] = set()
    result: list[T] = []
    for item in data:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


# Test queue
q = BoundedQueue[int](capacity=3)
assert q.is_empty()
q.enqueue(10)
q.enqueue(20)
q.enqueue(30)
assert q.is_full()
assert q.dequeue() == 10
assert not q.is_full()

# Test deduplication
raw_stream = [1, 2, 2, 5, 8, 4, 4, 8, 1, 9]
deduped = deduplicate_stable(raw_stream)
assert deduped == [1, 2, 5, 8, 4, 9]
print("Queue and deduplication lab exercises passed successfully!")